# Phase 4: ABX Listening Test — Results Analysis

**Purpose:** Analyze subjective ABX test results to identify the transparency point per codec.

### Workflow
1. Run the ABX test web app (GitHub Codespaces) with 5+ participants
2. Download the CSV results from each participant
3. Upload all CSVs to this notebook
4. Notebook computes accuracy, statistical significance, and transparency estimates

In [ ]:
!pip install pandas numpy scipy matplotlib seaborn --quiet

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import glob, os, io, warnings

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.facecolor'] = '#0a0a0f'
plt.rcParams['axes.facecolor'] = '#0e0e18'
plt.rcParams['text.color'] = '#e0e0e0'
plt.rcParams['axes.labelcolor'] = '#e0e0e0'
plt.rcParams['xtick.color'] = '#888'
plt.rcParams['ytick.color'] = '#888'
sns.set_palette(['#4a9eff', '#4aff9e', '#ff4a6a', '#ffaa4a', '#a855f7'])

print("\u2705 Dependencies ready.")

## 1. Load ABX test results

Upload all participant CSV files (exported from the ABX web app).

In [ ]:
# Option A: Upload from Colab
# from google.colab import files
# uploaded = files.upload()

# Option B: Load from directory
RESULTS_DIR = "abx_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

csv_files = glob.glob(f"{RESULTS_DIR}/*.csv")

if len(csv_files) == 0:
    print("No CSV files found. Generating demo data for pipeline testing...")
    print("(Replace with real ABX test data before final analysis!)\n")

    # Generate synthetic demo data
    np.random.seed(42)
    rows = []
    participants = ['P01', 'P02', 'P03', 'P04', 'P05']
    codecs = ['mp3', 'aac', 'opus']
    bitrates = [64, 96, 128, 192]
    categories = ['music', 'speech', 'ambient']

    for pid in participants:
        trial = 0
        for codec in codecs:
            for bitrate in bitrates:
                trial += 1
                # Higher bitrate = harder to distinguish = lower accuracy
                # Opus generally better than AAC better than MP3
                codec_bonus = {'opus': 0.15, 'aac': 0.10, 'mp3': 0.0}[codec]
                prob_correct = min(0.95, 0.5 + (256 - bitrate) / 512 - codec_bonus)
                correct = np.random.random() < prob_correct
                cat = np.random.choice(categories)

                rows.append({
                    'trial': trial,
                    'participant': pid,
                    'codec': codec,
                    'bitrate': bitrate,
                    'category': cat,
                    'answer': 'A' if np.random.random() < 0.5 else 'B',
                    'correct': correct,
                    'confidence': np.random.choice([1, 2, 3], p=[0.2, 0.3, 0.5]),
                    'listens_a': np.random.randint(1, 5),
                    'listens_b': np.random.randint(1, 5),
                    'listens_x': np.random.randint(1, 5),
                    'timestamp': f'2026-04-25T{10+trial//4}:{trial%60:02d}:00Z'
                })

    df = pd.DataFrame(rows)
    df.to_csv(f"{RESULTS_DIR}/demo_all_participants.csv", index=False)
    csv_files = [f"{RESULTS_DIR}/demo_all_participants.csv"]
    print(f"  Generated {len(rows)} demo trials across {len(participants)} participants.")

# Load all CSVs
dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True)

print(f"\n\u2705 Loaded {len(df)} trials from {df['participant'].nunique()} participant(s)")
print(f"   Codecs: {sorted(df['codec'].unique())}")
print(f"   Bitrates: {sorted(df['bitrate'].unique())} kbps")
df.head()

## 2. Overall accuracy

Is overall performance significantly above chance (50%)?

In [ ]:
total = len(df)
correct = df['correct'].sum()
accuracy = correct / total
p_value = stats.binomtest(correct, total, 0.5, alternative='greater').pvalue

print(f"Overall: {correct}/{total} correct ({accuracy*100:.1f}%)")
print(f"Binomial test vs chance (50%): p = {p_value:.4f}")
print(f"{'Significant (p < 0.05) \u2014 listeners CAN distinguish' if p_value < 0.05 else 'NOT significant \u2014 near transparency overall'}")

# Per-participant
print(f"\nPer-participant accuracy:")
for pid, grp in df.groupby('participant'):
    acc = grp['correct'].mean()
    n = len(grp)
    p = stats.binomtest(int(grp['correct'].sum()), n, 0.5, alternative='greater').pvalue
    sig = '*' if p < 0.05 else ''
    print(f"  {pid}: {acc*100:.0f}% ({int(grp['correct'].sum())}/{n}) p={p:.3f} {sig}")

## 3. Per-codec accuracy by bitrate

This is the key analysis — at which bitrate does each codec reach transparency?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

codecs = sorted(df['codec'].unique())
colors = {'mp3': '#ff4a6a', 'aac': '#4a9eff', 'opus': '#4aff9e'}

summary_data = []

for idx, codec in enumerate(codecs):
    ax = axes[idx]
    codec_df = df[df['codec'] == codec]

    bitrates = sorted(codec_df['bitrate'].unique())
    accs = []
    cis = []

    for br in bitrates:
        br_df = codec_df[codec_df['bitrate'] == br]
        n = len(br_df)
        k = int(br_df['correct'].sum())
        acc = k / n if n > 0 else 0.5
        # Wilson confidence interval
        z = 1.96
        p_hat = acc
        denom = 1 + z**2 / n
        center = (p_hat + z**2 / (2*n)) / denom
        margin = z * np.sqrt((p_hat*(1-p_hat) + z**2/(4*n)) / n) / denom
        accs.append(acc)
        cis.append(margin)

        p_val = stats.binomtest(k, n, 0.5, alternative='greater').pvalue
        summary_data.append({
            'codec': codec, 'bitrate': br, 'accuracy': acc,
            'n': n, 'correct': k, 'p_value': p_val,
            'significant': p_val < 0.05
        })

    ax.errorbar(bitrates, [a*100 for a in accs], yerr=[c*100 for c in cis],
                marker='o', linewidth=2, markersize=8, capsize=4,
                color=colors.get(codec, '#888'), label=codec.upper())

    # Chance line
    ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.axhspan(0, 62.5, alpha=0.05, color='green')  # ~transparency zone
    ax.text(min(bitrates), 55, 'Transparency zone', fontsize=8, color='green', alpha=0.6)

    ax.set_xlabel('Bitrate (kbps)')
    ax.set_ylabel('Accuracy (%)')
    ax.set_title(f'{codec.upper()}', fontsize=14, fontweight='bold', color=colors.get(codec))
    ax.set_ylim(30, 105)
    ax.grid(True, alpha=0.1)

plt.suptitle('ABX Accuracy by Codec and Bitrate', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('abx_accuracy_by_codec.png', dpi=300, bbox_inches='tight',
            facecolor='#0a0a0f', edgecolor='none')
plt.show()

# Summary table
summary_df = pd.DataFrame(summary_data)
print("\nDetailed Results:")
print(summary_df.to_string(index=False))

## 4. Transparency point estimation

The transparency point is the bitrate where accuracy drops to statistically indistinguishable from chance (50%).

In [ ]:
print("Transparency Point Estimation")
print("=" * 50)
print("(Bitrate where p > 0.05, i.e., accuracy not significantly above chance)\n")

for codec in sorted(df['codec'].unique()):
    codec_summary = summary_df[summary_df['codec'] == codec].sort_values('bitrate')
    transparent_br = None

    for _, row in codec_summary.iterrows():
        if not row['significant']:  # p > 0.05
            transparent_br = int(row['bitrate'])
            break

    if transparent_br:
        print(f"  {codec.upper():<6} => Transparent at {transparent_br} kbps "
              f"(accuracy {codec_summary[codec_summary['bitrate']==transparent_br]['accuracy'].values[0]*100:.0f}%, "
              f"p={codec_summary[codec_summary['bitrate']==transparent_br]['p_value'].values[0]:.3f})")
    else:
        max_br = codec_summary['bitrate'].max()
        print(f"  {codec.upper():<6} => NOT transparent at any tested bitrate "
              f"(still distinguishable at {max_br} kbps)")

print("\nNote: With small sample sizes, these estimates have wide confidence intervals.")
print("More participants = more statistical power = more reliable transparency estimates.")

## 5. Confidence-weighted analysis

Trials where participants were guessing vs. confident.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

conf_labels = {1: 'Guessing', 2: 'Somewhat sure', 3: 'Confident'}
for conf in [1, 2, 3]:
    conf_df = df[df['confidence'] == conf]
    if len(conf_df) == 0:
        continue
    bitrates = sorted(conf_df['bitrate'].unique())
    accs = [conf_df[conf_df['bitrate'] == br]['correct'].mean() * 100 for br in bitrates]
    ax.plot(bitrates, accs, marker='o', linewidth=2, label=conf_labels[conf])

ax.axhline(50, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xlabel('Bitrate (kbps)')
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy by Confidence Level')
ax.legend()
ax.grid(True, alpha=0.1)
ax.set_ylim(30, 105)
plt.tight_layout()
plt.savefig('abx_by_confidence.png', dpi=300, bbox_inches='tight',
            facecolor='#0a0a0f', edgecolor='none')
plt.show()

## 6. MOS-like scoring

Convert ABX results to a Mean Opinion Score approximation for the report.

In [ ]:
# Map: correct + confident = lower quality (more distinguishable)
# MOS 5 = indistinguishable, MOS 1 = very different
def abx_to_mos(row):
    if not row['correct']:
        return 5.0  # Couldn't tell apart = excellent
    conf_penalty = {1: 0.5, 2: 1.0, 3: 1.5}[row['confidence']]
    return max(1.0, 5.0 - conf_penalty)

df['mos_approx'] = df.apply(abx_to_mos, axis=1)

fig, ax = plt.subplots(figsize=(10, 5))

for codec in sorted(df['codec'].unique()):
    codec_df = df[df['codec'] == codec]
    bitrates = sorted(codec_df['bitrate'].unique())
    mos_vals = [codec_df[codec_df['bitrate'] == br]['mos_approx'].mean() for br in bitrates]
    ax.plot(bitrates, mos_vals, marker='o', linewidth=2, markersize=8,
            color=colors.get(codec, '#888'), label=codec.upper())

ax.axhline(4.0, color='green', linestyle=':', linewidth=0.8, alpha=0.5)
ax.text(max(df['bitrate'].unique()) + 2, 4.05, 'Good (MOS 4)', fontsize=8, color='green', alpha=0.6)
ax.axhline(3.0, color='orange', linestyle=':', linewidth=0.8, alpha=0.5)
ax.text(max(df['bitrate'].unique()) + 2, 3.05, 'Fair (MOS 3)', fontsize=8, color='orange', alpha=0.6)

ax.set_xlabel('Bitrate (kbps)')
ax.set_ylabel('MOS (approx)')
ax.set_title('Approximate Mean Opinion Score from ABX Results')
ax.set_ylim(1, 5.5)
ax.legend()
ax.grid(True, alpha=0.1)
plt.tight_layout()
plt.savefig('abx_mos_approx.png', dpi=300, bbox_inches='tight',
            facecolor='#0a0a0f', edgecolor='none')
plt.show()

## 7. Export for report

In [ ]:
import zipfile

zip_name = "phase4_abx_analysis.zip"
with zipfile.ZipFile(zip_name, 'w') as z:
    for f in glob.glob("*.png"):
        z.write(f)
    # Save summary table
    summary_df.to_csv("abx_summary_table.csv", index=False)
    z.write("abx_summary_table.csv")

print(f"\u2705 Exported {zip_name} ({os.path.getsize(zip_name)/1024:.0f} KB)")
# from google.colab import files
# files.download(zip_name)

## Summary

### For the report
- Use the **per-codec accuracy plot** as the main subjective result figure
- Report transparency points with confidence intervals
- Note sample size limitations
- Compare subjective transparency points with Phase 2 objective ODG estimates
- Discuss any discrepancies between objective and subjective thresholds